<a href="https://colab.research.google.com/github/ashwin-aggarwal/buildWithJev/blob/main/TestingJev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# !pip install langchain-typesafe
from langchain_typesafe import TypeSafeClassifier, Noul, Choice, Score
from google.colab import userdata

TYPESAFE_API_KEY = userdata.get("TYPESAFE_API_KEY")
classifier = TypeSafeClassifier(api_key=TYPESAFE_API_KEY)

incoming_ticket = """
The deploy pipeline completely failed twice. Production users are hitting 500 errors
on the checkout page. I tried rolling back but the database migration is stuck.
Please assign this to the infrastructure team immediately.
"""

questions = {
    "is_urgent": Noul(
        instructions="Does this need immediate, on-call engineering attention?",
    ),
    "routing_target": Choice(
        options=["frontend", "backend", "infrastructure", "security"],
        instructions="Which team should this ticket be routed to?",
        criteria={
            "frontend": "UI, client-side rendering, browser or styling issues",
            "backend": "Application logic, APIs, server-side bugs",
            "infrastructure": "Deploys, CI/CD, databases, migrations, hosting, outages",
            "security": "Vulnerabilities, auth, data exposure, suspicious access",
        },
    ),
    "severity_level": Score(
        scale=5,
        instructions="Rate the severity from 1 (minor) to 5 (catastrophic breakdown)",
        criteria=[
            "1: Cosmetic issue, no functional impact",
            "2: Minor bug with an easy workaround",
            "3: Degraded functionality affecting some users",
            "4: Major feature broken for many users, no workaround",
            "5: Production outage or data at risk, core flows down",
        ],
    ),
}

response = classifier.invoke({"state": incoming_ticket, "questions": questions})
answers = response.model_dump()["answers"]
urgent_p = answers["is_urgent"]["noul"]
route = answers["routing_target"]
sev = answers["severity_level"]

print("Classification Results:\n")
print(f"Is Urgent?  {urgent_p >= 0.5} (p={urgent_p:.2f})")
print(f"Route to:   {route['choice']} (confidence={route['confidence']:.2f})")
print(f"Severity:   {sev['score'] + 1:.2f} / 5 (confidence={sev['confidence']:.2f})")

Classification Results:

Is Urgent?  True (p=0.96)
Route to:   infrastructure (confidence=1.00)
Severity:   4.98 / 5 (confidence=0.98)


In [ ]:
tickets = [
    {
        "id": "INC-001",
        "text": """
        Production checkout is returning 500 errors for most users.
        The deployment failed twice and rollback is blocked by a
        database migration.
        """,
    },
    {
        "id": "INC-002",
        "text": """
        The button on the settings page is slightly misaligned on mobile.
        No functionality is affected.
        """,
    },
    {
        "id": "INC-003",
        "text": """
        Our API is returning intermittent 503 errors after today's release.
        The service has restarted several times and latency is increasing.
        """,
    },
    {
        "id": "INC-004",
        "text": """
        We detected suspicious authentication requests from an unknown
        IP range against several customer accounts.
        """,
    },
]

In [ ]:
results = []

for ticket in tickets:
    response = classifier.invoke({
        "state": ticket["text"],
        "questions": questions,
    })

    results.append({
        "id": ticket["id"],
        "is_urgent": response["is_urgent"],
        "routing_target": response["routing_target"],
        "severity_level": response["severity_level"],
    })

results


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df


In [5]:
def route_ticket(result):
    if result["is_urgent"]:
        priority = "P0/P1"
    else:
        priority = "normal"

    return {
        "team": result["routing_target"],
        "priority": priority,
    }


for result in results:
    print(result["id"], "→", route_ticket(result))


NameError: name 'results' is not defined